In [15]:
import numpy as np
import argparse

class IceCreamRobot:
    def __init__(self, size=5):
        self.size = size
        self.actions = ['Up', 'Down', 'Left', 'Right']
        self.action_map = {'Up': (-1, 0), 'Down': (1, 0), 'Left': (0, -1), 'Right': (0, 1)}
        self.state = (0, 0)  # Robot starts at top-left
        self.face = '😊'  # Initial happy face
        # Define grid items: 'I' for ice cream, 'C' for cactus, '.' for empty
        self.grid = [['.' for _ in range(size)] for _ in range(size)]
        self.grid[2][2] = 'I'  # Ice cream at (2,2)
        self.grid[4][4] = 'I'  # Ice cream at (4,4)
        self.grid[1][3] = 'C'  # Cactus at (1,3)
        self.grid[3][1] = 'C'  # Cactus at (3,1)

    def reset(self):
        self.state = (0, 0)
        self.face = '😊'
        return self.state

    def lick(self):
        """Determine what the robot licks and update its face."""
        x, y = self.state
        item = self.grid[x][y]
        if item == 'I':
            self.face = '😊'  # Happy for ice cream
            return "ice cream"
        elif item == 'C':
            self.face = '😣'  # Sad for cactus
            return "cactus"
        else:
            self.face = '😊'  # Default happy for empty
            return "nothing"

    def deterministic_step(self, action):
        """Move deterministically in the action direction."""
        move = self.action_map[action]
        new_state = (self.state[0] + move[0], self.state[1] + move[1])
        # Check boundaries
        new_state = (
            max(0, min(new_state[0], self.size - 1)),
            max(0, min(new_state[1], self.size - 1))
        )
        self.state = new_state
        return self.state

    def probabilistic_step(self, action):
        """Move probabilistically: 80% intended, 10% adjacent directions."""
        probs = {'Up': 0.1, 'Down': 0.1, 'Left': 0.1, 'Right': 0.1}
        probs[action] = 0.8  # 80% chance for intended action
        # Remove opposite direction
        opposites = {'Up': 'Down', 'Down': 'Up', 'Left': 'Right', 'Right': 'Left'}
        del probs[opposites[action]]
        probs = {k: v / sum(probs.values()) for k, v in probs.items()}  # Normalize
        actual_action = np.random.choice(list(probs.keys()), p=list(probs.values()))
        return self.deterministic_step(actual_action)

    def display_grid(self):
        """Display the grid with robot's position, ice cream, and cactus icons."""
        display = [['.' for _ in range(self.size)] for _ in range(self.size)]
        for i in range(self.size):
            for j in range(self.size):
                if self.grid[i][j] == 'I':
                    display[i][j] = '🍨'  # Ice cream icon
                elif self.grid[i][j] == 'C':
                    display[i][j] = '🌵'  # Cactus icon
                else:
                    display[i][j] = '.'
        x, y = self.state
        display[x][y] = self.face  # Robot's face
        print("\nGrid State:")
        for row in display:
            print(' '.join(row))

def simulate_robot(policy="deterministic", steps=5):
    """Simulate the robot with the specified policy."""
    np.random.seed(42)  # For reproducibility
    robot = IceCreamRobot(size=5)
    print(f"\nSimulating Robot with {policy} Policy:")
    print(f"Initial state: {robot.reset()}, Face: {robot.face}, Licked: {robot.lick()}")
    robot.display_grid()
    
    for step in range(steps):
        action = np.random.choice(robot.actions)  # Random action for simplicity
        if policy.lower() == "deterministic":
            new_state = robot.deterministic_step(action)
        else:  # Probabilistic
            new_state = robot.probabilistic_step(action)
        licked = robot.lick()
        print(f"\nStep {step + 1}: Action = {action}, New state = {new_state}, "
              f"Face = {robot.face}, Licked = {licked}")
        robot.display_grid()


In [19]:
# Test deterministic policy with 5 steps
simulate_robot(policy="deterministic", steps=100)


Simulating Robot with deterministic Policy:
Initial state: (0, 0), Face: 😊, Licked: nothing

Grid State:
😊 . . . .
. . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 1: Action = Left, New state = (0, 0), Face = 😊, Licked = nothing

Grid State:
😊 . . . .
. . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 2: Action = Right, New state = (0, 1), Face = 😊, Licked = nothing

Grid State:
. 😊 . . .
. . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 3: Action = Up, New state = (0, 1), Face = 😊, Licked = nothing

Grid State:
. 😊 . . .
. . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 4: Action = Left, New state = (0, 0), Face = 😊, Licked = nothing

Grid State:
😊 . . . .
. . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 5: Action = Left, New state = (0, 0), Face = 😊, Licked = nothing

Grid State:
😊 . . . .
. . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 6: Action = Right, New state = (0, 1), Face = 😊, Licked = nothing

Grid State:
. 😊 . . .
. . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 7: Action = Up, New state = (0, 1)

In [21]:
# Test probabilistic policy with 30 steps
simulate_robot(policy="probabilistic", steps=30)


Simulating Robot with probabilistic Policy:
Initial state: (0, 0), Face: 😊, Licked: nothing

Grid State:
😊 . . . .
. . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 1: Action = Left, New state = (0, 0), Face = 😊, Licked = nothing

Grid State:
😊 . . . .
. . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 2: Action = Left, New state = (0, 0), Face = 😊, Licked = nothing

Grid State:
😊 . . . .
. . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 3: Action = Up, New state = (0, 0), Face = 😊, Licked = nothing

Grid State:
😊 . . . .
. . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 4: Action = Down, New state = (1, 0), Face = 😊, Licked = nothing

Grid State:
. . . . .
😊 . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 5: Action = Left, New state = (1, 0), Face = 😊, Licked = nothing

Grid State:
. . . . .
😊 . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 6: Action = Up, New state = (0, 0), Face = 😊, Licked = nothing

Grid State:
😊 . . . .
. . . 🌵 .
. . 🍨 . .
. 🌵 . . .
. . . . 🍨

Step 7: Action = Right, New state = (0, 1),